# Spike-in positive control

In [ ]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy.stats import mannwhitneyu, rankdata
from scipy.sparse import issparse
from tqdm import tqdm
import textwrap
import muon as mu

warnings.filterwarnings("ignore")

_PRIMARY_DIR = "../.experiment_data/primary"
sys.path.append(os.path.join(_PRIMARY_DIR, "benchmarking"))

from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader

print("imports OK")


In [ ]:
GROUP_COLORS = {
    "B-cell markers": "#c12075",
    "DNA damage/stress": "#003754",
    "ER stress/UPR": "#4c626c",
    "pDC-specific": "#70acc0",
    "T cell exhaustion": "#723c5d",
    "Cytotoxic effectors": "#00695c",
    "Chemokines": "#e65100",
}
ANN_COLOR = "#c12075"
OTHER_COLOR = "#9bbfcc"
IMM_ANN_COLOR = "#e87722"
IMM_OTHER_COLOR = "#a8d8b0"
BROAD_ANN_COLOR = "#7b5ea7"
BROAD_OTHER_COLOR = "#c5b9d9"

plt.rcParams.update(
    {
        "font.size": 7,
        "axes.titlesize": 7.5,
        "axes.labelsize": 7,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "legend.fontsize": 5.5,
        "figure.dpi": 150,
        "font.family": "sans-serif",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

OUTDIR = "outputs/figS27_neurips2021_cite_bmmc_spike_in"
os.makedirs(OUTDIR, exist_ok=True)
print("OUTDIR:", OUTDIR)


## Section 1: GO Resources

In [ ]:
RESOURCES_DIR = os.path.join(
    _PRIMARY_DIR, "preprocessing", "resources", "ebi_ensembl_go_mappings"
)

obodag = GODag(os.path.join(RESOURCES_DIR, "go-basic.obo"))

ensembl2go = []
for gaf_name in [
    "goa_human_ensembl_gene_mapping_2024-08-08.gaf",
    "goa_human_isoform_ensembl_gene_mapping_2024-08-08.gaf",
    "goa_human_rna_ensembl_gene_mapping_2024-08-08.gaf",
]:
    objanno = GafReader(os.path.join(RESOURCES_DIR, gaf_name))
    ensembl2go.append(objanno.get_ns2assc()["BP"])

print(f"GO DAG: {len(obodag)} terms")
print(f"GAF files loaded: {len(ensembl2go)}")


In [ ]:
IMMUNE_ROOT = "GO:0002376"
immune_go_terms = set()
if IMMUNE_ROOT in obodag:
    immune_go_terms = obodag[IMMUNE_ROOT].get_all_children()
    immune_go_terms.add(IMMUNE_ROOT)
print(f"Immune GO terms (GO:0002376 + descendants): {len(immune_go_terms)}")


## Section 2: Experiment Configuration

In [ ]:
EXPERIMENTS = [
    (
        "ENSG00000177455",
        "CD19",
        "pDC",
        5.0,
        "B-cell markers",
        "B cell marker, absent in pDC, negative control",
    ),
    ("ENSG00000156738", "MS4A1", "pDC", 5.0, "B-cell markers", "CD20, B cell surface antigen"),
    ("ENSG00000196092", "PAX5", "CD8+ T naive", 5.0, "B-cell markers", "B cell lineage TF"),
    (
        "ENSG00000107447",
        "DNTT",
        "Naive CD20+ B IGKC+",
        5.0,
        "B-cell markers",
        "TdT, V(D)J recombination",
    ),
    ("ENSG00000004468", "CD38", "CD8+ T naive", 5.0, "B-cell markers", "Ecto-NAD+ase / ectoenzyme"),
    ("ENSG00000116717", "GADD45A", "pDC", 5.0, "DNA damage/stress", "DNA damage sensor"),
    (
        "ENSG00000124762",
        "CDKN1A",
        "pDC",
        5.0,
        "DNA damage/stress",
        "p21, CDK inhibitor / senescence",
    ),
    (
        "ENSG00000171791",
        "BCL2",
        "CD8+ T naive",
        5.0,
        "DNA damage/stress",
        "Anti-apoptotic regulator",
    ),
    ("ENSG00000100219", "XBP1", "pDC", 5.0, "ER stress/UPR", "UPR master TF, lipid metabolism"),
    ("ENSG00000118217", "ATF6", "pDC", 5.0, "ER stress/UPR", "UPR sensor / ER stress TF"),
    ("ENSG00000105492", "SIGLEC6", "NK", 5.0, "pDC-specific", "pDC marker spiked in NK cells"),
    ("ENSG00000275302", "CCL4", "CD8+ T naive", 5.0, "Chemokines", "Chemokine CCL4 / MIP-1b"),
    ("ENSG00000271503", "CCL5", "CD8+ T naive", 5.0, "Chemokines", "RANTES / CCL5 chemokine"),
    ("ENSG00000198846", "TOX", "CD8+ T naive", 5.0, "T cell exhaustion", "Exhaustion master TF"),
    ("ENSG00000089692", "LAG3", "CD8+ T naive", 5.0, "T cell exhaustion", "Checkpoint receptor"),
    ("ENSG00000100453", "GZMB", "CD8+ T naive", 5.0, "Cytotoxic effectors", "Granzyme B"),
    ("ENSG00000180644", "PRF1", "CD8+ T naive", 5.0, "Cytotoxic effectors", "Perforin"),
]
print(f"{len(EXPERIMENTS)} experiments")


## Section 3: Load Experiment Data

In [ ]:
EXPERIMENTS_DIR = os.path.join(
    _PRIMARY_DIR,
    "experiments_new",
    "sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_"
    "mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_"
    "patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0",
    "default",
)
SPIKE_DIR = os.path.join(EXPERIMENTS_DIR, "spike_in_experiments")
CALC_DIR = os.path.join(EXPERIMENTS_DIR, "calc_go_terms_cell_type")

print("SPIKE_DIR exists:", os.path.isdir(SPIKE_DIR))
print("CALC_DIR  exists:", os.path.isdir(CALC_DIR))

def load_experiment(ensg_id, spiked_ct, strength):
    '''Load spike-in and baseline results.'''
    fname = (
        "cell_type_vs_rest_res_unp_act_mask_False_flip_False_go_terms_perturbation_"
        f"{ensg_id}_spikein_{ensg_id}_{strength}_mult_{spiked_ct}"
        "_evaluated_graph_csv_with_ci.pkl"
    )
    path = os.path.join(SPIKE_DIR, fname)
    if not os.path.exists(path):
        print(f"  MISSING: {fname}")
        return None
    with open(path, "rb") as fh:
        spk = pickle.load(fh)["expression"]["expression"]

    ct_results = {}
    go_index = go_names = None
    for ct, vals in spk.items():
        slug = ct.replace(" ", "_")
        bl_path = os.path.join(
            CALC_DIR,
            f"expression_cell_type_vs_rest_{slug}_"
            "res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl",
        )
        if not os.path.exists(bl_path):
            continue
        bl = pd.read_pickle(bl_path)
        spk_roc = vals["roc"].astype(float)
        bl_roc = bl["predictors_label_valid_roc"].values.astype(float)
        if go_index is None:
            go_index = bl.index.values
            go_names = bl["name"].values
        ct_results[ct] = {"delta": bl_roc - spk_roc, "baseline_roc": bl_roc}
    return {"ct_results": ct_results, "go_index": go_index, "go_names": go_names}


In [ ]:
all_results = {}
for ensg_id, gene_name, spiked_ct, strength, group, desc in tqdm(EXPERIMENTS):
    res = load_experiment(ensg_id, spiked_ct, strength)
    if res is None:
        print(f"MISSING: {gene_name}")
        continue
    all_results[ensg_id] = {
        "gene_name": gene_name,
        "spiked_ct": spiked_ct,
        "strength": strength,
        "group": group,
        "desc": desc,
        **res,
    }
print(f"\nLoaded {len(all_results)}/{len(EXPERIMENTS)} experiments")


## Section 4: GO Annotation Ancestor Propagation

In [ ]:
def get_go_annotations_propagated(ensg_id, model_go_set):
    '''Return set of model GO terms annotated to ensg_id, with ancestor propagation.'''
    direct_gos = set()
    for e2g in ensembl2go:
        if ensg_id in e2g:
            direct_gos.update(e2g[ensg_id])

    all_gos = set(direct_gos)
    for g in direct_gos:
        if g in obodag:
            all_gos.update(obodag[g].get_all_parents())

    n_direct = len(direct_gos & model_go_set)
    ann_in_model = all_gos & model_go_set
    return ann_in_model, n_direct, len(ann_in_model)

go_annotations = {}
ann_stats_table = {}

print("{:<12s} {:>10} {:>14}  (GO terms in model)".format("Gene", "n_direct", "n_propagated"))
print("-" * 42)
for ensg_id, gene_name, *_ in EXPERIMENTS:
    if ensg_id not in all_results:
        continue
    model_gos = set(all_results[ensg_id]["go_index"])
    ann_set, n_dir, n_prop = get_go_annotations_propagated(ensg_id, model_gos)
    go_annotations[ensg_id] = ann_set
    ann_stats_table[ensg_id] = {"gene": gene_name, "n_direct": n_dir, "n_propagated": n_prop}
    print(f"{gene_name:12s} {n_dir:10d} {n_prop:14d}")


In [ ]:
immune_annotations = {}

print("{:<12s} {:>14} {:>16}".format("Gene", "n_ann_immune", "n_immune_model"))
print("-" * 46)
for ensg_id, gene_name, *_ in EXPERIMENTS:
    if ensg_id not in all_results:
        continue
    model_gos = set(all_results[ensg_id]["go_index"])
    immune_model = model_gos & immune_go_terms
    ann_immune = go_annotations.get(ensg_id, set()) & immune_model
    immune_annotations[ensg_id] = ann_immune
    print(f"{gene_name:12s} {len(ann_immune):14d} {len(immune_model):16d}")


In [ ]:
BROAD_PARENTS = {"GO:0002376", "GO:0065007", "GO:0009987"}
_sample_ensg = next(iter(all_results))
_model_go_set = set(all_results[_sample_ensg]["go_index"])

broad_bio_gos = set()
for _pid in BROAD_PARENTS:
    if _pid in obodag:
        broad_bio_gos.add(_pid)
        broad_bio_gos.update(obodag[_pid].get_all_children())
broad_bio_gos &= _model_go_set

broad_bio_annotations = {}
print("{:<12s} {:>12} {:>14}".format("Gene", "n_ann_broad", "n_broad_model"))
print("-" * 42)
for ensg_id, gene_name, *_ in EXPERIMENTS:
    if ensg_id not in all_results:
        continue
    broad_ann = go_annotations.get(ensg_id, set()) & broad_bio_gos
    broad_bio_annotations[ensg_id] = broad_ann
    print(f"{gene_name:12s} {len(broad_ann):12d} {len(broad_bio_gos):14d}")
print(f"\nBroad biology filter: {len(broad_bio_gos)} model terms")
print("(descendants of GO:0002376 ∪ GO:0065007 ∪ GO:0009987)")


## Section 5: Statistical Analysis

In [ ]:
def compute_stats(ensg_id):
    """One-sided Mann-Whitney U + effect size for one experiment."""
    res = all_results[ensg_id]
    spiked_ct = res["spiked_ct"]
    if spiked_ct not in res["ct_results"]:
        return None
    delta = res["ct_results"][spiked_ct]["delta"]
    go_index = res["go_index"]
    annotated = go_annotations.get(ensg_id, set())
    ann_mask = np.array([g in annotated for g in go_index], dtype=bool)
    n_ann = int(ann_mask.sum())
    abs_delta = np.abs(delta)

    if n_ann == 0:
        return {
            "n_ann": 0,
            "p": np.nan,
            "effect": np.nan,
            "med_rank_pct": np.nan,
            "mean_ann": np.nan,
            "mean_all": float(np.mean(abs_delta)),
            "max_ann": np.nan,
        }

    ann_d = abs_delta[ann_mask]
    other_d = abs_delta[~ann_mask]
    _, p_val = mannwhitneyu(ann_d, other_d, alternative="greater")
    effect = float(np.mean(ann_d)) / (float(np.mean(other_d)) + 1e-12)
    med_rank_pct = float(np.median(rankdata(abs_delta)[ann_mask])) / len(abs_delta) * 100

    return {
        "n_ann": n_ann,
        "p": float(p_val),
        "effect": float(effect),
        "med_rank_pct": float(med_rank_pct),
        "mean_ann": float(np.mean(ann_d)),
        "mean_all": float(np.mean(abs_delta)),
        "max_ann": float(np.max(ann_d)),
    }

summary_rows = []
for ensg_id, gene_name, spiked_ct, strength, group, desc in EXPERIMENTS:
    if ensg_id not in all_results:
        continue
    s = compute_stats(ensg_id)
    if s is None:
        continue
    ad = ann_stats_table.get(ensg_id, {})
    summary_rows.append(
        {
            "gene": gene_name,
            "ensg": ensg_id,
            "group": group,
            "spiked_ct": spiked_ct,
            "strength": strength,
            "n_direct": ad.get("n_direct", 0),
            "n_ann": s["n_ann"],
            "p_value": s["p"],
            "effect_size": s["effect"],
            "med_rank_pct": s["med_rank_pct"],
            "mean_ann": s["mean_ann"],
            "mean_all": s["mean_all"],
            "max_ann": s["max_ann"],
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(
    summary_df[
        [
            "gene",
            "group",
            "spiked_ct",
            "n_direct",
            "n_ann",
            "effect_size",
            "p_value",
            "med_rank_pct",
        ]
    ]
    .sort_values("p_value")
    .round(4)
)


In [ ]:
print("=" * 105)
print(
    "{:^105}".format(
        "FULL RESULTS: SparseVI spike-in analysis (ancestor-propagated GO annotations)"
    )
)
print("=" * 105)
hdr = "{:<10s} {:<22s} {:<26s} {:>5} {:>5} {:>8} {:>11} {:>6}".format(
    "Gene", "Group", "Spiked CT", "n_dir", "n_ann", "effect", "p-value", "med%"
)
print(hdr)
print("-" * 105)
for _, r in summary_df.sort_values("p_value").iterrows():
    p_s = f"{r.p_value:.3e}" if not np.isnan(r.p_value) else "(no ann)"
    eff = f"{r.effect_size:.2f}x" if not np.isnan(r.effect_size) else "N/A"
    med = f"{r.med_rank_pct:.1f}%" if not np.isnan(r.med_rank_pct) else "N/A"
    print(
        f"{r.gene:10s} {r.group:22s} {r.spiked_ct:26s} "
        f"{int(r.n_direct):5d} {int(r.n_ann):5d} {eff:>8} {p_s:>11} {med:>6}"
    )
print("=" * 105)


In [ ]:
print("\nTOP ANNOTATED GO TERMS PER EXPERIMENT (top 8 by |ΔAUROC| in spiked CT)\n")
for ensg_id, gene_name, spiked_ct, *_ in EXPERIMENTS:
    if ensg_id not in all_results:
        continue
    res = all_results[ensg_id]
    if spiked_ct not in res["ct_results"]:
        continue
    delta = res["ct_results"][spiked_ct]["delta"]
    dauroc = -delta
    go_index = res["go_index"]
    go_names_a = res["go_names"]
    annotated = go_annotations.get(ensg_id, set())
    ann_mask = np.array([g in annotated for g in go_index], dtype=bool)
    n_ann_imm = len(immune_annotations.get(ensg_id, set()))

    print(
        f"{gene_name} ({ensg_id}) -> {spiked_ct}  "
        f"[n_ann={ann_mask.sum()}, n_ann_immune={n_ann_imm}]"
    )
    if not ann_mask.any():
        print("  (no annotated GO terms in model)")
        print()
        continue
    ann_idx = np.where(ann_mask)[0]
    top_ann = ann_idx[np.argsort(np.abs(dauroc[ann_idx]))[::-1][:8]]
    for ii in top_ann:
        nm = (go_names_a[ii] or go_index[ii])[:65]
        imm = " [immune]" if go_index[ii] in immune_go_terms else ""
        print(f"  {go_index[ii]}  ΔAUROC={dauroc[ii]:+.4f}  {nm}{imm}")
    print()


## Section 6: Expression Analysis

In [ ]:
H5MU_PATH = os.path.join(
    _PRIMARY_DIR,
    "preprocessing",
    "datasets",
    "neurips2021_cite_BMMC",
    "data",
    "neurips2021_cite_BMMC.h5mu",
)
print("Loading h5mu ...")
mdata = mu.read(H5MU_PATH)
rna = mdata["rna"]
print(f"RNA shape: {rna.shape[0]:,} cells x {rna.shape[1]:,} genes")

gene_stable_ids = rna.var["gene_stable_id"].tolist()
ensg2col = {gid: i for i, gid in enumerate(gene_stable_ids) if isinstance(gid, str)}

X_counts = rna.layers["counts"]
ct_obs = rna.obs["cell_type"].values
print(f"Cell types in dataset: {len(set(ct_obs))}")


In [ ]:
expr_stats = {}

print(
    "{:<10s} {:<26s} {:>10} {:>10} {:>8}".format(
        "Gene", "Spiked CT", "mean_cnt", "pct_expr", "n_cells"
    )
)
print("-" * 70)
for ensg_id, gene_name, spiked_ct, *_ in EXPERIMENTS:
    col_idx = ensg2col.get(ensg_id)
    if col_idx is None:
        print(f"{gene_name}: not found in var (ENSG not in gene_stable_id)")
        expr_stats[ensg_id] = {"mean_count": np.nan, "pct_expr": np.nan, "n_cells": 0}
        continue
    ct_mask = ct_obs == spiked_ct
    n_cells = int(ct_mask.sum())
    if n_cells == 0:
        print(f"{gene_name}: cell type {spiked_ct!r} not found in obs")
        expr_stats[ensg_id] = {"mean_count": np.nan, "pct_expr": np.nan, "n_cells": 0}
        continue
    col_vec = X_counts[ct_mask, col_idx]
    counts_arr = (
        np.asarray(col_vec.todense()).flatten()
        if issparse(col_vec)
        else np.asarray(col_vec).flatten()
    )
    mean_c = float(counts_arr.mean())
    pct_e = float((counts_arr > 0).mean() * 100)
    expr_stats[ensg_id] = {"mean_count": mean_c, "pct_expr": pct_e, "n_cells": n_cells}
    print(f"{gene_name:10s} {spiked_ct:26s} {mean_c:10.3f} {pct_e:9.1f}% {n_cells:8,}")


In [ ]:
summary_df["mean_count_spiked_ct"] = [
    expr_stats.get(e, {}).get("mean_count", np.nan) for e in summary_df["ensg"]
]
summary_df["pct_expr_spiked_ct"] = [
    expr_stats.get(e, {}).get("pct_expr", np.nan) for e in summary_df["ensg"]
]
display(
    summary_df[
        [
            "gene",
            "spiked_ct",
            "mean_count_spiked_ct",
            "pct_expr_spiked_ct",
            "n_ann",
            "effect_size",
            "p_value",
        ]
    ]
    .sort_values("mean_count_spiked_ct", ascending=False)
    .round(3)
)


## Section 7: Figure Generation

In [ ]:
def plot_ranked_bar(
    ensg_id,
    n_top=25,
    filter_mode="all",
    save_path=None,
    ax=None,
    panel_label=None,
    show_legend=True,
    force_annotated=True,
):
    '''Ranked horizontal bar chart of ΔAUROC for top GO terms in the spiked CT.'''
    res = all_results[ensg_id]
    gene_name = res["gene_name"]
    spiked_ct = res["spiked_ct"]
    go_index = res["go_index"]
    go_names_a = res["go_names"]
    delta = res["ct_results"][spiked_ct]["delta"]
    dauroc = -delta

    if filter_mode == "immune":
        filter_gos = immune_go_terms
        annotated = immune_annotations.get(ensg_id, set())
        ann_color = IMM_ANN_COLOR
        oth_color = IMM_OTHER_COLOR
    elif filter_mode == "broad":
        filter_gos = broad_bio_gos
        annotated = broad_bio_annotations.get(ensg_id, set())
        ann_color = BROAD_ANN_COLOR
        oth_color = BROAD_OTHER_COLOR
    else:
        filter_gos = None
        annotated = go_annotations.get(ensg_id, set())
        ann_color = ANN_COLOR
        oth_color = OTHER_COLOR

    ann_mask = np.array([g in annotated for g in go_index], dtype=bool)

    if filter_gos is not None:
        valid_mask = np.array([g in filter_gos for g in go_index], dtype=bool)
        if valid_mask.sum() == 0:
            print(f"{gene_name}: no {filter_mode} GO terms -> skipping")
            return None
    else:
        valid_mask = np.ones(len(go_index), dtype=bool)

    abs_d = np.where(valid_mask, np.abs(dauroc), -1.0)
    top_idx = np.argsort(abs_d)[::-1][:n_top]
    if force_annotated:
        ann_valid = np.where(ann_mask & valid_mask)[0]
        combined = list(dict.fromkeys(list(top_idx) + list(ann_valid)))
        combined = [i for i in combined if valid_mask[i]]
        combined = sorted(combined, key=lambda i: np.abs(dauroc[i]), reverse=True)[:n_top]
    else:
        combined = [i for i in top_idx if valid_mask[i]]

    if not combined:
        return None

    labels, values, bar_colors = [], [], []
    for i in combined:
        name = go_names_a[i] or go_index[i]
        if len(name) > 58:
            name = name[:56] + ".."
        ia = go_index[i] in annotated
        labels.append(name)
        values.append(dauroc[i])
        bar_colors.append(ann_color if ia else oth_color)

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7, max(3.5, len(values) * 0.22 + 0.8)))

    y_pos = list(range(len(values)))
    ax.barh(
        y_pos,
        list(reversed(values)),
        color=list(reversed(bar_colors)),
        edgecolor="white",
        linewidth=0.2,
        height=0.82,
    )
    ax.set_yticks(y_pos)
    ax.set_yticklabels(list(reversed(labels)), fontsize=5.5)
    ax.axvline(0, color="black", lw=0.5)
    ax.set_xlabel("ΔAUROC  (spiked − baseline)", fontsize=6.5)

    ann_list = [go_index[i] in annotated for i in combined]
    for tick, ia in zip(ax.get_yticklabels(), reversed(ann_list)):
        if ia:
            tick.set_fontweight("bold")
            tick.set_color(ann_color)

    row = summary_df[summary_df["ensg"] == ensg_id]
    if not row.empty and not np.isnan(row.p_value.values[0]):
        r = row.iloc[0]
        stat_str = (
            f"n={int(ann_mask[valid_mask].sum())}  effect={r.effect_size:.1f}x  p={r.p_value:.2e}"
        )
    else:
        stat_str = "n=0 (no test)"

    stats_y = 0.13 if show_legend else 0.01
    ax.annotate(
        stat_str,
        xy=(0.98, stats_y),
        xycoords="axes fraction",
        ha="right",
        va="bottom",
        fontsize=5,
        color="#555",
    )

    if show_legend:
        patches = [
            mpatches.Patch(color=ann_color, label="annotated (propagated)"),
            mpatches.Patch(color=oth_color, label="other GO terms"),
        ]
        ax.legend(
            handles=patches,
            fontsize=4.5,
            loc="lower right",
            frameon=False,
            bbox_to_anchor=(0.99, 0.01),
        )

    if standalone:
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight", transparent=True)
            if save_path.endswith(".png"):
                plt.savefig(save_path[: -len(".png")] + ".svg", format="svg", bbox_inches="tight", transparent=True)
            print(f"  saved {save_path}")
        plt.show()
        return fig
    return ax


## Section 8: Per-Gene Individual Figures

In [ ]:
for ensg_id, gene_name, spiked_ct, strength, group, desc in EXPERIMENTS:
    if ensg_id not in all_results:
        continue
    if spiked_ct not in all_results[ensg_id]["ct_results"]:
        print(f"WARNING: {spiked_ct} not evaluated for {gene_name}")
        continue

    row = summary_df[summary_df["ensg"] == ensg_id]
    r = row.iloc[0] if not row.empty else None

    print(f'\n{"-"*60}')
    print(f"{gene_name}  ({ensg_id})  |  {group}")
    if r is not None and not np.isnan(r.p_value):
        print(
            f"  n_ann={int(r.n_ann)}  direct={int(r.n_direct)}  "
            f"effect={r.effect_size:.2f}x  p={r.p_value:.3e}  "
            f"expression={r.mean_count_spiked_ct:.2f} ({r.pct_expr_spiked_ct:.1f}%)"
        )

    print(f"  [all-GO]")
    plot_ranked_bar(ensg_id, n_top=25, filter_mode="all", force_annotated=True, save_path=None)

    print(f"  [all-GO pure top-25]")
    plot_ranked_bar(ensg_id, n_top=25, filter_mode="all", force_annotated=False, save_path=None)

    n_imm = len(immune_annotations.get(ensg_id, set()))
    if n_imm >= 1:
        print(f"  [immune filter]  n_ann_immune={n_imm}")
        plot_ranked_bar(ensg_id, n_top=25, filter_mode="immune", save_path=None)
    else:
        print(f"  [immune filter]  skipped (0 annotated immune terms)")

    n_broad = len(broad_bio_annotations.get(ensg_id, set()))
    if n_broad >= 1:
        print(f"  [broad filter]   n_ann_broad={n_broad}")
        plot_ranked_bar(ensg_id, n_top=25, filter_mode="broad", save_path=None)
    else:
        print(f"  [broad filter]   skipped (0 annotated broad terms)")


## Section 9: Overview Figure for All 19 Experiments

In [ ]:
sig_thresh = 0.05
df_plot = summary_df[summary_df["n_ann"] > 0].copy()

df_a = df_plot.sort_values("effect_size", ascending=True)

df_b = df_plot.sort_values("p_value", ascending=False)
pvals_clip = np.clip(-np.log10(df_b["p_value"].fillna(1) + 1e-15), 0, 20)

fig_c, axes = plt.subplots(1, 2, figsize=(10, 5.2))
fig_c.subplots_adjust(top=0.88, bottom=0.18, left=0.18, right=0.97)

axes[0].barh(
    df_a["gene"],
    df_a["effect_size"],
    color=[GROUP_COLORS.get(g, "#9d9c9c") for g in df_a["group"]],
    edgecolor="white",
    linewidth=0.3,
    height=0.75,
)
axes[0].axvline(1.0, color="black", lw=0.8, ls="--")
axes[0].set_xlabel("Effect size  (mean|ΔAUROC| ann / mean|ΔAUROC| all)", fontsize=6.5)
axes[0].tick_params(labelsize=6)

axes[1].barh(
    df_b["gene"],
    pvals_clip,
    color=[GROUP_COLORS.get(g, "#9d9c9c") for g in df_b["group"]],
    edgecolor="white",
    linewidth=0.3,
    height=0.75,
)
axes[1].axvline(-np.log10(sig_thresh), color="gray", lw=0.8, ls="--")
axes[1].set_xlabel("−log₁₀(p-value)", fontsize=6.5)
axes[1].tick_params(labelsize=6)

patches_c = [
    mpatches.Patch(color=c, label=g)
    for g, c in GROUP_COLORS.items()
    if g in df_plot["group"].values
]
fig_c.legend(
    handles=patches_c,
    loc="lower center",
    ncol=4,
    fontsize=5.5,
    bbox_to_anchor=(0.5, 0.0),
    frameon=False,
)

p_c = os.path.join(OUTDIR, "spikein_v5_FigS_overview.png")
p_c_pdf = os.path.join(OUTDIR, "spikein_v5_FigS_overview.pdf")
p_c_svg = os.path.join(OUTDIR, "spikein_v5_FigS_overview.svg")
plt.savefig(p_c, dpi=300, bbox_inches="tight", transparent=True)
plt.savefig(p_c_pdf, bbox_inches="tight")
plt.savefig(p_c_svg, format="svg", bbox_inches="tight")
print(f"  saved {p_c}")
plt.show()


## Section 10: Expression Level vs. Effect Size Dot Plot

In [ ]:
try:
    from adjustText import adjust_text

    _has_adjusttext = True
except ImportError:
    _has_adjusttext = False
    print("adjusttext not installed; labels may overlap")
from collections import defaultdict

dot_df = summary_df[summary_df["n_ann"] > 0].copy()
dot_df["log10_mean"] = np.log10(dot_df["mean_count_spiked_ct"].clip(lower=0) + 0.01)
dot_df["neg_log10_p"] = np.clip(-np.log10(dot_df["p_value"].fillna(1) + 1e-15), 0, 20)
dot_df["dot_size"] = (dot_df["neg_log10_p"].clip(lower=0.5) * 22 + 30).clip(lower=30)

print("Expression level vs. enrichment strength dot plot")
print("Key takeaway: enrichment strength is NOT correlated with baseline expression")

CT_ABBREV = {
    "pDC": "pDC",
    "CD8+ T naive": "CD8+T",
    "Naive CD20+ B IGKC+": "B",
    "NK": "NK",
}

fig, ax = plt.subplots(figsize=(11.0, 7.5))
fig.subplots_adjust(right=0.58)

all_x = dot_df["log10_mean"].values
all_y = dot_df["effect_size"].values

for grp, gdf in dot_df.groupby("group"):
    c = GROUP_COLORS.get(grp, "#888")
    ax.scatter(
        gdf["log10_mean"],
        gdf["effect_size"],
        s=gdf["dot_size"],
        c=c,
        alpha=0.85,
        edgecolors="white",
        linewidths=0.6,
        zorder=3,
    )

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
ax_bbox = ax.get_window_extent(renderer=renderer)
xlim, ylim = ax.get_xlim(), ax.get_ylim()
x_range = xlim[1] - xlim[0]
y_range = ylim[1] - ylim[0]
pts_to_data_x = x_range * fig.dpi / (ax_bbox.width * 72.0)
pts_to_data_y = y_range * fig.dpi / (ax_bbox.height * 72.0)

CLUSTER_X_RADIUS = 0.15
CLUSTER_Y_LIMIT = 6.0
clearance = y_range * 0.04

BELOW_GENES = {"DNTT", "CDKN1A"}
below_clearance = y_range * 0.015

x_starts = list(all_x)
y_starts = []
for i, r in dot_df.iterrows():
    r_pts = np.sqrt(r["dot_size"] / np.pi)
    own_top = r["effect_size"] + r_pts * pts_to_data_y
    mask = (
        (np.abs(dot_df["log10_mean"] - r["log10_mean"]) < CLUSTER_X_RADIUS)
        & (dot_df["effect_size"] > r["effect_size"])
        & ((dot_df["effect_size"] - r["effect_size"]) < CLUSTER_Y_LIMIT)
    )
    if mask.any():
        neighbor_tops = (
            dot_df.loc[mask, "effect_size"]
            + np.sqrt(dot_df.loc[mask, "dot_size"] / np.pi) * pts_to_data_y
        )
        ys = neighbor_tops.max() + clearance
    else:
        ys = own_top + clearance
    y_starts.append(ys)

cluster_groups = defaultdict(list)
for idx, ys in enumerate(y_starts):
    cluster_groups[round(ys, 2)].append(idx)
for key, members in cluster_groups.items():
    if len(members) < 2:
        continue
    members_sorted = sorted(members, key=lambda i: x_starts[i])
    cx = np.mean([x_starts[i] for i in members_sorted])
    step = 0.25
    n = len(members_sorted)
    offsets = np.linspace(-(n - 1) / 2.0, (n - 1) / 2.0, n) * step
    for i, m in enumerate(members_sorted):
        x_starts[m] = cx + offsets[i]

texts_above = []
text_dot_pairs = []
for idx, (i, r) in enumerate(dot_df.iterrows()):
    ct = CT_ABBREV.get(r["spiked_ct"], r["spiked_ct"][:5])
    label = f"{r['gene']} ({ct})"
    color = GROUP_COLORS.get(r["group"], "#444")
    dx, dy, dot_sz = r["log10_mean"], r["effect_size"], r["dot_size"]

    if r["gene"] in BELOW_GENES:
        r_pts = np.sqrt(dot_sz / np.pi)
        by = dy - r_pts * pts_to_data_y - below_clearance
        t = ax.text(
            dx,
            by,
            label,
            fontsize=5.5,
            color=color,
            fontweight="bold",
            ha="center",
            va="top",
            zorder=5,
        )
    else:
        t = ax.text(
            x_starts[idx],
            y_starts[idx],
            label,
            fontsize=5.5,
            color=color,
            fontweight="bold",
            ha="center",
            va="bottom",
            zorder=5,
        )
        texts_above.append(t)
    text_dot_pairs.append((t, dx, dy, dot_sz))

if _has_adjusttext:
    adjust_text(
        texts_above,
        ax=ax,
        expand=(1.5, 1.0),
        force_text=(3.0, 0.05),
        force_static=(0, 0),
        force_pull=(0.3, 3.0),
        pull_threshold=300,
        iter_lim=600,
    )

fig.canvas.draw()
ax_bbox2 = ax.get_window_extent(renderer=renderer)
for t in texts_above:
    tb = t.get_window_extent(renderer=renderer)
    if tb.x0 < ax_bbox2.x0 + 2:
        shift_data = (ax_bbox2.x0 + 4 - tb.x0) / (ax_bbox2.width / x_range)
        tx, ty = t.get_position()
        t.set_position((tx + shift_data, ty))

for t, dx, dy, dot_sz in text_dot_pairs:
    tx, ty = t.get_position()
    r_pts = np.sqrt(dot_sz / np.pi)
    ax.annotate(
        "",
        xy=(dx, dy),
        xytext=(tx, ty),
        arrowprops=dict(arrowstyle="-", color="#aaa", lw=0.6, shrinkA=2, shrinkB=r_pts + 2),
    )

ax.axhline(1.0, color="#999", lw=0.7, ls="--")
ax.set_xlabel(
    "log₁₀(mean raw UMI count in spiked cell type + 0.01)\n"
    "← near-absent in spiked CT                   highly expressed →",
    fontsize=8,
)
ax.set_ylabel("Enrichment effect size\n(mean|ΔAUROC| annotated / mean|ΔAUROC| all)", fontsize=8)
ax.grid(True, alpha=0.2, lw=0.5)

from matplotlib.lines import Line2D

group_handles = [
    mpatches.Patch(color=GROUP_COLORS.get(g, "#888"), label=g)
    for g in dot_df["group"].unique()
    if g in GROUP_COLORS
]
size_legend = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="#888",
        markersize=np.sqrt(np.clip(-np.log10(p), 0.5, 20) * 22 + 30) / 4,
        label=lbl,
        alpha=0.85,
    )
    for p, lbl in [(0.05, "p = 0.05"), (0.001, "p = 0.001"), (1e-10, "p = 1e−10")]
]
leg1 = ax.legend(
    handles=group_handles,
    fontsize=6,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    title=None,
)
ax.add_artist(leg1)
ax.legend(
    handles=size_legend,
    fontsize=6,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.01, 0.75),
    title="Dot size = −log₁₀(p)",
    title_fontsize=6.5,
)

p_dot = os.path.join(OUTDIR, "spikein_v5_expr_dot.png")
plt.savefig(p_dot, dpi=300, bbox_inches="tight", transparent=True)
plt.savefig(os.path.join(OUTDIR, "spikein_v5_expr_dot.svg"), format="svg", bbox_inches="tight", transparent=True)
print(f"  saved {p_dot}")
plt.show()


## Section 11: Combined Supplementary Figure

In [ ]:
EXCLUDED_FROM_PANEL = {"BCL2"}
EXCLUDED_GROUPS_FROM_PANEL = {"pDC-specific"}

valid = summary_df[
    (summary_df["n_ann"] > 0)
    & (~summary_df["gene"].isin(EXCLUDED_FROM_PANEL))
    & (~summary_df["group"].isin(EXCLUDED_GROUPS_FROM_PANEL))
].copy()

best_per_group = (
    valid.sort_values("p_value")
    .drop_duplicates(subset="group", keep="first")
    .sort_values("p_value")
    .reset_index(drop=True)
)

print("Panel genes (one per biological group, by lowest p-value):")
for _, r in best_per_group.iterrows():
    print(f"  {r.gene:<10s}  group={r.group:<22s}  p={r.p_value:.2e}  effect={r.effect_size:.1f}x")


In [ ]:
n_panels = len(best_per_group)
ncols = 2
nrows = (n_panels + 1) // 2

print("Combined panel figure (per-group representatives):")
for _, br in best_per_group.iterrows():
    print(f"  {br.gene}  ({br.group}  ·  spiked into {br.spiked_ct})")

fig2, axes2 = plt.subplots(nrows, ncols, figsize=(11, nrows * 3.8))
axes2 = np.array(axes2).flatten()

for i, (_, br) in enumerate(best_per_group.iterrows()):
    ensg_id = br["ensg"]
    gene_name = br["gene"]
    spiked_ct = br["spiked_ct"]
    if ensg_id not in all_results or spiked_ct not in all_results[ensg_id]["ct_results"]:
        axes2[i].set_visible(False)
        continue
    print(f"  plotting {gene_name}...")
    plot_ranked_bar(ensg_id, n_top=20, filter_mode="all", ax=axes2[i], show_legend=False)
    axes2[i].set_title(f'{gene_name}  ({br["group"]})\nSpiked into: {spiked_ct}', fontsize=7, pad=4)

for j in range(n_panels, len(axes2)):
    axes2[j].set_visible(False)

shared_patches = [
    mpatches.Patch(color=ANN_COLOR, label="annotated GO terms (propagated)"),
    mpatches.Patch(color=OTHER_COLOR, label="other GO terms"),
]
fig2.legend(
    handles=shared_patches,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=2,
    fontsize=7.5,
    frameon=False,
)

plt.tight_layout(h_pad=2.5, w_pad=1.2)
plt.subplots_adjust(bottom=0.06)

p2 = os.path.join(OUTDIR, "spikein_v5_combined_all.png")
p2_pdf = os.path.join(OUTDIR, "spikein_v5_combined_all.pdf")
p2_svg = os.path.join(OUTDIR, "spikein_v5_combined_all.svg")
plt.savefig(p2, dpi=300, bbox_inches="tight", transparent=True)
plt.savefig(p2_pdf, bbox_inches="tight")
plt.savefig(p2_svg, format="svg", bbox_inches="tight")
print(f"  saved {p2}")
plt.show()

print("\nSaving individual panel figures for Figma...")
for i, (_, br) in enumerate(best_per_group.iterrows()):
    ensg_id = br["ensg"]
    gene_name = br["gene"]
    spiked_ct = br["spiked_ct"]
    if ensg_id not in all_results or spiked_ct not in all_results[ensg_id]["ct_results"]:
        continue
    print(f"  {gene_name}")
    fig_i, ax_i = plt.subplots(figsize=(5.5, 3.8))
    plot_ranked_bar(ensg_id, n_top=20, filter_mode="all", ax=ax_i, show_legend=True)
    ax_i.set_title(f'{gene_name}  ({br["group"]})\nSpiked into: {spiked_ct}', fontsize=7, pad=14)
    ax_i.annotate(
        chr(97 + i),
        xy=(-0.02, 1.08),
        xycoords="axes fraction",
        fontsize=13,
        fontweight="bold",
        va="bottom",
        ha="right",
    )
    plt.tight_layout()
    p_i = os.path.join(OUTDIR, f"spikein_v5_panel_{gene_name}.png")
    plt.savefig(p_i, dpi=300, bbox_inches="tight", transparent=True)
    plt.savefig(os.path.join(OUTDIR, f"spikein_v5_panel_{gene_name}.svg"), format="svg", bbox_inches="tight", transparent=True)
    print(f"    saved {p_i}")
    plt.close()


In [ ]:
import csv

table_rows = []
for _, r in summary_df.iterrows():
    ensg_id = r["ensg"]
    top5_terms = []
    if ensg_id in all_results:
        res = all_results[ensg_id]
        spiked_ct = res["spiked_ct"]
        if spiked_ct in res["ct_results"]:
            go_index = res["go_index"]
            go_names_a = res["go_names"]
            delta = res["ct_results"][spiked_ct]["delta"]
            dauroc = -delta
            annotated = go_annotations.get(ensg_id, set())
            ann_indices = [i for i, g in enumerate(go_index) if g in annotated]
            ann_indices_sorted = sorted(ann_indices, key=lambda i: abs(dauroc[i]), reverse=True)[:5]
            for i in ann_indices_sorted:
                gname = go_names_a[i] or go_index[i]
                top5_terms.append(f"{gname} (ΔAUROC={dauroc[i]:+.4f})")

    table_rows.append(
        {
            "gene": r["gene"],
            "ensembl_id": r["ensg"],
            "group": r["group"],
            "spiked_cell_type": r["spiked_ct"],
            "spike_multiplier": r["strength"],
            "n_direct_ann": int(r["n_direct"]),
            "n_propagated_ann": int(r["n_ann"]),
            "effect_size": round(r["effect_size"], 4),
            "p_value": f"{r['p_value']:.4e}",
            "med_rank_pct": round(r["med_rank_pct"], 1),
            "mean_raw_count": round(r.get("mean_count_spiked_ct", float("nan")), 4),
            "pct_expressing": round(r.get("pct_expr_spiked_ct", float("nan")), 1),
            "top5_annotated_go_terms": " | ".join(top5_terms),
        }
    )

table_df = pd.DataFrame(table_rows).sort_values("p_value")

table_path = os.path.join(OUTDIR, "spikein_v5_supplementary_table.csv")
table_df.to_csv(table_path, index=False)
print(f"Supplementary table saved: {table_path}")
print(f"{len(table_df)} experiments")
display(table_df.round(4))
